# HW03 · Bài 3 · Linear SVM

Thực hiện ba thí nghiệm trên slide 82 và 95. Giữ cố định Linear SVM và tập kiểm thử, thay đổi cách biểu diễn văn bản.

Phân loại cảm xúc trên dữ liệu IMDB của Stanford. Nhãn 0 là tiêu cực và nhãn 1 là tích cực. Cả năm notebook dùng chung mẫu 200 review lấy phân tầng từ tập test với seed 42 để so sánh.

## Nạp và tiền xử lý dữ liệu

Đọc review có nhãn từ tập IMDB gốc, sắp xếp theo tên tệp để việc lấy mẫu có thể lặp lại và bỏ thẻ HTML xuống dòng trước khi đưa review vào mô hình.

In [ ]:
import re
import shutil
import tarfile
from pathlib import Path
from urllib.request import urlopen

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

SEED = 42
SAMPLE_SIZE = 200
DATA_URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
ARCHIVE = Path("data/aclImdb_v1.tar.gz")
LABELS = {"neg": 0, "pos": 1}

if not ARCHIVE.exists():
    ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    partial = ARCHIVE.with_suffix(".tmp")
    try:
        with urlopen(DATA_URL, timeout=60) as response, partial.open("wb") as file:
            shutil.copyfileobj(response, file)
        partial.replace(ARCHIVE)
    finally:
        partial.unlink(missing_ok=True)


def load_imdb(splits):
    records = {split: [] for split in splits}
    with tarfile.open(ARCHIVE, "r|gz") as archive:
        for member in archive:
            parts = member.name.split("/")
            if not member.isfile() or len(parts) != 4 or parts[0] != "aclImdb":
                continue
            _, split, label, filename = parts
            if split not in records or label not in LABELS or not filename.endswith(".txt"):
                continue
            with archive.extractfile(member) as file:
                text = file.read().decode("utf-8")
            records[split].append((member.name, LABELS[label], text))

    datasets = {}
    for split, rows in records.items():
        if len(rows) != 25_000:
            raise ValueError(f"Tập {split} có {len(rows)} mẫu, dự kiến 25.000.")
        rows.sort(key=lambda row: row[0])
        datasets[split] = (
            [row[2] for row in rows],
            np.array([row[1] for row in rows], dtype=np.int32),
        )
    return datasets


def clean_text(text):
    return re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE).strip()

In [ ]:
data = load_imdb(("train", "test"))
X_train, y_train = data["train"]
X_test, y_test = data["test"]

shared_indices, _ = train_test_split(
    np.arange(len(y_test)), train_size=SAMPLE_SIZE,
    stratify=y_test, random_state=SEED,
)
shared_indices.sort()
print(f"IMDB test: {len(y_test):,} mẫu | Mẫu so sánh: {len(shared_indices)}")

## Độ đo đánh giá

Macro-F1 lấy trung bình F1 của hai nhãn. Accuracy và Macro-F1 được báo cáo trên thang từ 0 đến 1.

In [ ]:
def scores(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(
            y_true, y_pred, labels=[0, 1], average="macro", zero_division=0,
        ),
    }


def evaluate(model_name, predictions):
    return [
        {
            "Mô hình": model_name, "Tập đánh giá": "IMDB test", "Số mẫu": len(y_test),
            **scores(y_test, predictions),
        },
        {
            "Mô hình": model_name, "Tập đánh giá": "Mẫu so sánh",
            "Số mẫu": len(shared_indices),
            **scores(y_test[shared_indices], predictions[shared_indices]),
        },
    ]

## Ba cấu hình đặc trưng

BoW đếm số lần từ xuất hiện. TF-IDF gán trọng số cho từ đơn. TF-IDF `(1, 2)` dùng cả từ đơn và cụm hai từ liên tiếp. Cả ba cấu hình giữ các từ phủ định như *not*.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

vectorizers = {
    "BoW": CountVectorizer(min_df=2),
    "TF-IDF": TfidfVectorizer(min_df=2),
    "TF-IDF + n-gram (1, 2)": TfidfVectorizer(min_df=2, ngram_range=(1, 2)),
}
train_texts = [clean_text(text) for text in X_train]
test_texts = [clean_text(text) for text in X_test]

rows = []
for name, vectorizer in vectorizers.items():
    model = make_pipeline(vectorizer, LinearSVC(C=1.0, random_state=SEED, max_iter=10_000))
    model.fit(train_texts, y_train)
    predictions = model.predict(test_texts)
    rows.extend(evaluate(name, predictions))

## Kết quả và nhận xét

Điền bảng ở slide 95 bằng kết quả toàn tập test. Cấu hình nào có Macro-F1 cao nhất, chênh BoW bao nhiêu điểm phần trăm? Dựa vào trọng số TF-IDF và thông tin từ bigram để giải thích.

In [ ]:
results = pd.DataFrame(rows)
full = results[results["Tập đánh giá"] == "IMDB test"].set_index("Mô hình")
metrics = ["Accuracy", "Macro-F1"]
deltas = (full[metrics] - full.loc["BoW", metrics]) * 100
deltas.columns = ["Δ Accuracy (điểm %)", "Δ Macro-F1 (điểm %)"]

display(full.join(deltas).round(4))
display(results[results["Tập đánh giá"] == "Mẫu so sánh"].reset_index(drop=True).round(4))

## Tài liệu

[IMDB gốc](https://ai.stanford.edu/~amaas/data/sentiment/) · [LinearSVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html)